In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, get_scheduler, BitsAndBytesConfig, AutoModelForCausalLM
from tqdm.auto import tqdm

In [5]:
batch_size = 16
lr = 5e-5
epochs = 3
temperature = 2.0
alpha_soft = 0.5
max_len = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# Load dataset
raw = load_dataset("tweet_eval", "sentiment")

In [7]:
label_feature = raw["train"].features["label"]

In [8]:
print("Label names: ", label_feature.names)

Label names:  ['negative', 'neutral', 'positive']


In [9]:
# Train data
train = raw['train'].shuffle(seed=42)

In [10]:
# Validation data
val = raw['validation']

In [11]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [12]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=max_len)

In [13]:
# Tokenize and remove original text column
tokenized = {}

In [14]:
tokenized['train'] = train.map(tokenize, batched=True, remove_columns=['text'])
tokenized['validation'] = val.map(tokenize, batched=True, remove_columns=['text'])

In [15]:
tokenized

{'train': Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 45615
 }),
 'validation': Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 2000
 })}

In [16]:
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

In [17]:
# Dataloaders
train_dl = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True, collate_fn=collator)

In [18]:
val_dl = DataLoader(tokenized['validation'], batch_size=batch_size, shuffle=False,collate_fn=collator)

In [19]:
num_labels = 3

In [20]:
teacher = AutoModelForSequenceClassification.from_pretrained("bert-large-uncased", num_labels=num_labels).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
student = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

In [ ]:
# Freeze teacher (no training)
for p in teacher.parameters():
    p.requires_grad = False


In [ ]:
teacher.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): L

In [ ]:
ce_loss = nn.CrossEntropyLoss()

In [ ]:
kl_loss = nn.KLDivLoss(reduction="batchmean")

In [ ]:
optimizer = optim.Adam(student.parameters(), lr=lr)

In [ ]:
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_dl) * epochs,
)

In [ ]:
def distill_epoch():
    student.train()
    pbar = tqdm(train_dl, desc="Train")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention = batch["attention_mask"].to(device)  
        labels = batch["labels"].to(device)

        # Teacher predictions (soft targets)
        with torch.no_grad():
            t_logits = teacher(input_ids, attention_mask=attention).logits
            t_soft = torch.softmax(t_logits / temperature, dim=1)

        # Student predictions
        s_logits = student(input_ids, attention_mask=attention).logits
        s_soft = torch.log_softmax(s_logits / temperature, dim=1)

        # Distillation + CE Loss
        loss_soft = kl_loss(s_soft, t_soft) * (temperature ** 2)
        loss_hard = ce_loss(s_logits, labels)
        loss = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

In [ ]:
def evaluate():
    student.eval()  
    correct = total = 0
    with torch.no_grad():
        for batch in val_dl:
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)  
            lbl = batch["labels"].to(device)
            out = student(ids, attention_mask=attn).logits
            pred = out.argmax(dim=1)
            correct += (pred == lbl).sum().item()
            total += lbl.size(0)
    return round(correct / total * 100, 2)

In [ ]:
for ep in range(1, epochs + 1):
    distill_epoch()
    acc = evaluate()
    print(f"Epoch {ep}/{epochs} | Validation Accuracy: {acc}%")

Train:   0%|          | 0/2851 [00:00<?, ?it/s]

Epoch 1/3 | Validation Accuracy: 72.4%


Train:   0%|          | 0/2851 [00:00<?, ?it/s]

Epoch 2/3 | Validation Accuracy: 71.9%


Train:   0%|          | 0/2851 [00:00<?, ?it/s]

Epoch 3/3 | Validation Accuracy: 72.9%


In [ ]:
student.save_pretrained("distilled_student_model")
tokenizer.save_pretrained("distilled_student_model")

('distilled_student_model/tokenizer_config.json',
 'distilled_student_model/special_tokens_map.json',
 'distilled_student_model/vocab.txt',
 'distilled_student_model/added_tokens.json',
 'distilled_student_model/tokenizer.json')

In [ ]:
# Load test set
test = load_dataset("tweet_eval", "sentiment", split="test[:500]")
tokenized_test = test.map(tokenize, batched=True, remove_columns=["text"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dl = DataLoader(tokenized_test, batch_size=batch_size, shuffle=False, collate_fn=collator)

In [ ]:
from sklearn.metrics import accuracy_score
import time

def predict_and_evaluate(model, name, test_dl):
    model.eval()
    all_preds, all_labels = [], []
    start_time = time.time()

    with torch.no_grad():
        for batch in test_dl:
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)  
            lbls = batch["labels"].to(device)

            logits = model(ids, attention_mask=attn).logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(lbls.cpu().tolist())  

    total_time = time.time() - start_time
    acc = accuracy_score(all_labels, all_preds) 
    avg_time = total_time / len(test_dl.dataset)

    print(f"\n {name}") 
    print(f" Accuracy: {acc*100:.2f}%")
    print(f" Total Inference Time: {total_time:.2f} sec")
    print(f" Avg Time per Sample: {avg_time:.4f} sec")
    return acc, total_time, avg_time

In [ ]:
# Compare teachere vs student
predict_and_evaluate(teacher, name="TEACHER (BERT-Large)", test_dl=test_dl)
predict_and_evaluate(student, name="STUDENT (Distilled-BERT)", test_dl=test_dl)


 TEACHER (BERT-Large)
 Accuracy: 22.40%
 Total Inference Time: 1.20 sec
 Avg Time per Sample: 0.0024 sec

 STUDENT (Distilled-BERT)
 Accuracy: 67.40%
 Total Inference Time: 0.43 sec
 Avg Time per Sample: 0.0009 sec


(0.674, 0.4258456230163574, 0.0008516912460327148)

## Big Model

In [31]:
teacher_id = "NousResearch/Llama-2-7b-chat-hf"
student_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [32]:
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

In [33]:
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

In [34]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# teacher = AutoModelForCausalLM.from_pretrained(
#     teacher_id,
#     device_map="auto",
#     quantization_config=bnb_config
# )

In [26]:
teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [27]:
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

In [28]:
student = AutoModelForCausalLM.from_pretrained(
    student_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [29]:
prompts = [
    "Explain why the sky is blue. ### The sky appears blue because molecules in Earth's atmosphere scatter sunlight, and blue light is scattered more than ...",
    "What is the capital of France? ### The capital of France is Paris.",
    "Write a short story about a robot and a cat. ### Once upon a time, a lonely robot found a stray cat. They became best friends, exploring the city toge...",
]

In [36]:
# Distillation Hyperparameters
temperature = 2.0
alpha_soft = 0.7
ce_loss = nn.CrossEntropyLoss(ignore_index=student_tokenizer.pad_token_id)
kl_loss = nn.KLDivLoss(reduction="batchmean")   
optimizer = optim.Adam(student.parameters(), lr=2e-5)

In [ ]:
for prompt in prompts:
    t_inputs = teacher_tokenizer(prompt, return_tensors="pt", padding=True).to(teacher.device)
    s_inputs = student_tokenizer(prompt, return_tensors="pt", padding=True).to(student.device)

    with torch.no_grad():
        t_logits = teacher(**t_inputs).logits[:, :-1, :]
        t_soft = torch.softmax(t_logits / temperature, dim=-1)

    s_logits = student(**s_inputs).logits[:, :-1, :]
    s_log_soft = torch.log_softmax(s_logits / temperature, dim=-1)

    labels = s_inputs["input_ids"][:, 1:].contiguous()

    loss_hard = ce_loss(s_logits.reshape(-1, s_logits.size(-1)), labels.reshape(-1))
    loss_soft = kl_loss(s_log_soft, t_soft) * (temperature ** 2)

    loss = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

    if torch.isnan(loss):
        print(" NaN detected on prompt:", prompt[: 50])
        continue

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Prompt: {prompt[:40]}..., Loss: {loss.item():.4f}")

Prompt: Explain why the sky is blue. ### The sky..., Loss: 77.0780
Prompt: What is the capital of France? ### The c..., Loss: 34.5168
Prompt: Write a short story about a robot and a ..., Loss: 66.7333
